In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# โมเดล CNN แบบง่าย
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, 1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 5 * 5, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

# ข้อมูล MNIST
transform = transforms.ToTensor()
train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

# อุปกรณ์
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleCNN().to(device)

# RMSProp Optimizer + StepLR Scheduler
optimizer = optim.RMSprop(model.parameters(), lr=0.01, alpha=0.9, eps=1e-08)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

# ลด LR ลงครึ่งหนึ่งทุก 2 epochs
# scheduler จะลด learning rate
# RMSprop จะไม่ลด learning rate อัตโนมัติ เพียงแค่ปรับ "น้ำหนักของการอัปเดต" โดยใช้ค่าเฉลี่ยของกราเดียนต์ แต่ค่า LR ที่ตั้งไว้ (lr=0.01) จะคงที่ตลอด
# ยกเว้น คุณใช้ scheduler ร่วมด้วย

criterion = nn.CrossEntropyLoss()




In [2]:
# Training Loop
total_epochs = 20
for epoch in range(total_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    scheduler.step()  # อัปเดต learning rate ทุก epoch

    # แสดงค่า learning rate ปัจจุบัน
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1:02d}/{total_epochs} | LR: {current_lr:.6f} | Loss: {running_loss:.4f}")

Epoch 01/20 | LR: 0.010000 | Loss: 226.3389
Epoch 02/20 | LR: 0.005000 | Loss: 70.2398
Epoch 03/20 | LR: 0.005000 | Loss: 36.2566
Epoch 04/20 | LR: 0.002500 | Loss: 33.4672
Epoch 05/20 | LR: 0.002500 | Loss: 19.1082
Epoch 06/20 | LR: 0.001250 | Loss: 16.0898
Epoch 07/20 | LR: 0.001250 | Loss: 9.6305
Epoch 08/20 | LR: 0.000625 | Loss: 8.6862
Epoch 09/20 | LR: 0.000625 | Loss: 6.1066
Epoch 10/20 | LR: 0.000313 | Loss: 5.6691
Epoch 11/20 | LR: 0.000313 | Loss: 4.6073
Epoch 12/20 | LR: 0.000156 | Loss: 4.2695
Epoch 13/20 | LR: 0.000156 | Loss: 3.8996
Epoch 14/20 | LR: 0.000078 | Loss: 3.7889


KeyboardInterrupt: 